# Step 1 - Diagnostics on real CA_1 results

Goal: understand where models fail before changing them. No retraining here -- everything reads the existing artifacts.

Design rules locked in for this layer:

* `lightgbm_quantile_p50` is a first-class model in every accuracy breakdown (it's currently the best-WAPE model on real CA_1).
* Demand segments are leakage-safe: each `(origin_date, id)` is classified using only `date <= origin_date`.
* All breakdowns happen on the matched grid (3 origins x 3049 ids x [1, 7, 14, 28] horizons = 36,588 rows per model).

All real logic lives in `seercast.diagnostics` and `seercast.visualization.diagnostic_plots`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from seercast.training.run_diagnostics import run as run_diagnostics

pd.options.display.float_format = '{:.4f}'.format
REPO_ROOT

## 1. Run the diagnostics pass

This reads every artifact, builds the matched-grid combined predictions frame, computes per-origin demand segments and lifecycle flags, and writes all CSVs + PNGs.

In [ ]:
result = run_diagnostics()
combined = result['combined']
segments = result['segments']
lifecycle = result['lifecycle']
breakdowns = result['breakdowns']
feature_importance = result['feature_importance']
overall_wape = result['overall_wape']
print('models on grid:', sorted(combined['model'].unique()))
print('rows per model:', combined.groupby('model').size().to_dict())

## 2. Overall WAPE leaderboard

In [ ]:
overall_wape.to_frame('WAPE_overall')

## 3. Demand segment distribution

Segments are per `(origin_date, id)`. The same id can move between segments at different origins (e.g. an item that accumulates history goes from `zero_heavy` to `intermittent`).

In [ ]:
segments['segment_at_origin'].value_counts().rename_axis('segment').to_frame('rows')

## 4. WAPE by horizon

How does each model degrade as horizon grows?

In [ ]:
by_h = breakdowns['horizon']
by_h.pivot(index='horizon', columns='model', values='WAPE')

## 5. WAPE by demand segment

**The most important diagnostic.** Different demand types have different best models -- intermittent and lumpy demand is famously hard for global ML models trained on aggregate loss.

In [ ]:
by_seg = breakdowns['segment_at_origin']
by_seg.pivot(index='segment_at_origin', columns='model', values='WAPE')

## 6. Bias by demand segment

Negative bias = underforecast on that segment; positive = overforecast. The reported quantile p50 negative bias (~ -0.17 overall) is the biggest known weakness; this view shows where it concentrates.

In [ ]:
by_seg.pivot(index='segment_at_origin', columns='model', values='Bias')

## 7. Performance on zero vs non-zero actuals

In [ ]:
if 'target_zero' in breakdowns:
    display(breakdowns['target_zero'].pivot(index='target_zero', columns='model', values='Bias'))
    display(breakdowns['target_zero'].pivot(index='target_zero', columns='model', values='WAPE'))

## 8. Pre-launch lifecycle share

How much of the matched grid is pre-launch (item didn't exist yet at the origin)?

In [ ]:
if 'is_active' in breakdowns:
    display(breakdowns['is_active'].pivot(index='is_active', columns='model', values='WAPE'))
    if 'is_active' in combined.columns:
        share = combined.groupby('is_active').size() / len(combined)
        print('rows by is_active:', share.to_dict())

## 9. Feature importance (point + quantile p50)

Importance averaged across the three per-origin boosters.

In [ ]:
feature_importance.head(20)

## 10. Worst under-forecasted items (for the current best WAPE model)

In [ ]:
from seercast.config import REPORTS_DIR
best_model = overall_wape.index[0]
worst_under = pd.read_csv(REPORTS_DIR / 'diagnostics' / f'worst_underforecast_{best_model}_top20.csv')
print(f'top 10 underforecasted ids for {best_model}:')
worst_under.head(10)

**Next:** review these results offline. After understanding where the model fails, we will decide which improvement step is most justified -- objective sweep, per-horizon models, calibration, or lifecycle handling.